# Chapter 2: Agent Control Flow

Estimated time: about 8 hours.

Prerequisites: Chapter 1 (the ReAct loop, mock tools, and `agentlib.llm_client` built
there are all reused here).

Interview category this chapter maps to: multi-agent architecture questions, such as when
multiple agents help vs. hurt, what a subagent actually is, and diagnosing which failure
mode ("it's slow," "it's stuck," "it's wrong") you're looking at from a symptom description
alone.


## Concept: from one employee to a small team

Chapter 1 built one employee working alone. Real systems usually don't stop there. This
chapter is about a small team, not "a pipeline": a supervisor who plans, a report who does
the work, and someone who checks it before it goes out. We'll use a consistent cast for this
team, reused for the rest of this course wherever a named multi-agent example is useful.
Jack is the planner, and breaks a task into subtasks. Bob is the worker, and executes the
subtasks, calling tools (and, in this chapter, dispatching subagents) as needed. Mike is the
critic and QA reviewer, and reviews Bob's output before it goes back to the user.

Naming them is deliberate. It makes a real multi-agent trace easier to read and discuss than
generic role labels, which matters once you're staring at a log with several agents in it.

### ReAct vs. plan-and-execute

Chapter 1's loop was pure ReAct: decide one step, act, observe, repeat, with no upfront
plan. Plan-and-execute is the alternative: produce a full plan first (Jack's job below), then
work through it, only replanning if something breaks. Neither is strictly better. ReAct
adapts more fluidly step-by-step but can wander; plan-and-execute is more predictable and
easier to review, but a bad plan stays bad until something forces a replan. Jack/Bob/Mike
below is a plan-and-execute pipeline with ReAct running *inside* each of Bob's subagent
dispatches, a common combination in practice, not an either/or choice.

### State machines for agents

An agent's control flow can also be modeled explicitly as a state machine: a fixed set of
states (planning, executing, reviewing, done) and transitions between them, rather than an
implicit loop buried in code. This is exactly what a framework like LangGraph makes concrete
(see the bridge section below). Nodes are states, edges are transitions, and some edges are
conditional on what happened in the node before them.

### Subagents: what actually makes something a "subagent"

Not just "another agent in a pipeline." The defining mechanic is context isolation: a
subagent gets a fresh, bounded context, executes its task independently, and returns a
compressed synthesis to the orchestrator, not its full transcript. That's what keeps the
orchestrator's own context from getting polluted by every worker's intermediate reasoning
(this chapter's build section implements this for real, and one of its break-it scenarios is
exactly what happens when that compression step is skipped).

There are four current patterns for how an orchestrator manages subagents, ordered by how
much lifecycle control the orchestrator keeps (Schmid, 2026; see `REFERENCES.md`). Inline
tool-call spawn is the simplest: calling a subagent looks identical to calling any other
tool, and it blocks until the subagent returns; this is what Bob does below. Fan-out gives
moderate control: multiple independent subagents dispatched in parallel, with results
collected once they all finish. Persistent agent pools give higher control: long-lived,
stateful workers reused across many tasks rather than spun up fresh each time. And
peer-to-peer teams give the least centralized control of all: agents message each other
directly, with no central dispatcher at all.

Supervisor-worker, which is what Jack/Bob/Mike already is, remains the current production
default. The existing pipeline below is the right foundation to build on, not something to
throw out in favor of something fancier.

### Skills: a companion concept, not the same thing

A skill is a small, self-contained instruction/tool package the orchestrator loads per task
to stay capable without bloating its own context. A rough rule of thumb: if it needs more
than a paragraph of documentation, it's probably two skills. Subagents isolate execution;
skills isolate capability. Production systems typically compose both: an orchestrator might
load a skill to know how to do something, then dispatch a subagent to actually do it in
isolation.


## Setup

In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import difflib
import json
import random

from agentlib import llm_client
from agentlib.grading import check
from agentlib.tracing import Tracer

random.seed(42)
print(f"Repo root on sys.path: {_repo_root}")
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")


Repo root on sys.path: /home/user/learning-agentic-ai
LLM_PROVIDER = 'anthropic', HAS_KEY = False


## Build: giving Bob a subagent to dispatch

A small, deterministic knowledge base (same idea as Chapter 1's `mock_search_tool`, redefined
here since this notebook is self-contained) so subagent work has something real to look up.

In [2]:
_TEAM_FACTS = {
    "anthropic founder": (
        "Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with "
        "several colleagues who had previously worked at OpenAI."
    ),
    "react pattern": (
        "ReAct interleaves reasoning traces with actions, letting a model plan and use "
        "tools within the same loop (Yao et al., 2022)."
    ),
}


def mock_search_tool(query: str) -> dict:
    q = query.lower()
    for key, fact in _TEAM_FACTS.items():
        if key in q or any(w in q for w in key.split()):
            return {"status": "ok", "result": fact}
    return {"status": "not_found", "result": f"No canned result for query: {query!r}"}


SUBAGENT_TOOLS = {"mock_search": mock_search_tool}


### The subagent loop itself

`run_subagent()` is the inline-tool-call-spawn pattern from the concept section, and it is
yours to implement. It starts from a completely fresh `messages` list -- nothing from Bob's
own conversation is visible to it -- runs its own small ReAct loop, and returns only the
final answer, the compressed synthesis, not the intermediate thoughts and tool observations
that built up along the way.

Both halves of that matter, and they are separate. *Fresh context in*: the subagent never
sees the parent's messages, which is what stops the parent's token cost being paid twice.
*Compressed result out*: only the answer crosses back, which is what stops the parent's
context filling with the subagent's scaffolding. Get either one wrong and the subagent stops
being worth spawning.

`leaky=True` exists only for this chapter's break-it section below: it returns the *entire*
raw transcript instead, which is the bug.

In [ ]:
def run_subagent(
    subtask: str,
    brain,
    tools: dict,
    max_iterations: int = 4,
    leaky: bool = False,
    verbose: bool = False,
):
    '''Run one subagent on one bounded subtask, in its own fresh context.

    Same ReAct shape as Chapter 1's run_agent(), with three differences:

    1. `messages` starts as [{"role": "user", "content": subtask}] and NOTHING else. Every
       call gets its own new list -- that is the context isolation.
    2. Return a (result, steps_used) tuple rather than a bare string.
    3. With leaky=True, return json.dumps(messages + [the final assistant turn]) instead of
       the compressed answer. That is the bug the break-it section below demonstrates.

    A tool the subagent doesn't have should produce {"status": "error", ...} as its
    observation rather than raising. Running out of iterations returns a message and
    max_iterations.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


run_subagent = check("ch02-subagent", run_subagent)

In [ ]:
def subagent_brain(messages: list) -> dict:
    '''Rule-based stand-in used inside each subagent's own fresh ReAct loop.'''
    observations = [m for m in messages if m["role"] == "observation"]
    if not observations:
        return {"action": "mock_search", "action_input": messages[0]["content"]}
    return {"action": "final_answer", "action_input": observations[-1]["content"].get("result", "")}


def real_subagent_brain_call(subtask: str) -> str:
    '''A real model call standing in for a subagent's work on one bounded subtask -- a
    single call_model round-trip against a fresh, isolated message list (context isolation
    with no extra plumbing, since there's nothing else in the list to isolate from).'''
    response = llm_client.call_model(
        messages=[{"role": "user", "content": subtask}],
        system="Answer the user's question directly and concisely in 1-2 sentences.",
        model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
    )
    return response.text


synthesis, steps_used = run_subagent("who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, verbose=True)
print(f"\nCompressed synthesis returned to Bob: {synthesis!r}")
print(f"(subagent used {steps_used} step(s) internally -- none of that trace crosses back, only the answer does)")

## Build: Jack plans, Bob dispatches, Mike reviews

Now the full team. Jack breaks the task into subtasks; Bob dispatches a subagent per subtask
(using `run_subagent()` above) and assembles a draft; Mike reviews the draft against a list of
facts it needs to cover. Every hop gets logged to a `Tracer` with a simulated latency value
(logged, not actually slept through, so this stays fast to run while still teaching realistic
per-hop timing).

Jack's decomposition is yours to write. The interface constraint is the interesting part: Bob
iterates over whatever Jack returns, so a plan is always a *list* of subtask strings, even
when the task only needs one. Hand back a bare string and Bob will iterate over its
characters.

In [ ]:
def fake_planner_brain(task: str) -> list:
    '''Jack, the planner -- rule-based stand-in. Recognize this chapter's one demo task
    (it mentions both Anthropic and ReAct) and split it into its two natural subtasks:
    "who founded anthropic?" and "what is the react pattern?".

    Anything else is already atomic: return it as a single-item list.

    Always a list of strings, never a bare string, and never empty -- Bob dispatches one
    subagent per element. Each subtask has to stand on its own, because the subagent that
    receives it starts from a fresh context and will see nothing but that string.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


fake_planner_brain = check("ch02-planner", fake_planner_brain)

In [ ]:
def real_planner_call(task: str) -> list:
    response = llm_client.call_model(
        messages=[{"role": "user", "content": task}],
        system=(
            "You are a planner. Break the user's task into 2-4 short, self-contained "
            "subtask strings. Respond with ONLY a JSON array of strings, nothing else."
        ),
        model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
    )
    try:
        subtasks = json.loads(response.text)
        if isinstance(subtasks, list) and all(isinstance(s, str) for s in subtasks):
            return subtasks
    except json.JSONDecodeError:
        pass
    return [task]


def fake_mike(draft: str, required_facts: list) -> dict:
    '''Mike, the critic -- rule-based stand-in: checks the draft mentions everything it's
    supposed to.'''
    missing = [f for f in required_facts if f.lower() not in draft.lower()]
    if missing:
        return {"verdict": "revise", "feedback": f"Missing coverage of: {', '.join(missing)}"}
    return {"verdict": "approved", "feedback": "Covers everything required."}


def real_critic_call(draft: str, required_facts: list) -> dict:
    response = llm_client.call_model(
        messages=[{
            "role": "user",
            "content": f"Draft:\n{draft}\n\nRequired facts to cover: {required_facts}",
        }],
        system=(
            "You are a critic reviewing a draft. If it covers all required facts, respond "
            "with exactly: APPROVED. Otherwise respond with: REVISE: <one sentence of "
            "feedback>."
        ),
        model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
    )
    text = response.text.strip()
    if text.upper().startswith("APPROVED"):
        return {"verdict": "approved", "feedback": text}
    return {"verdict": "revise", "feedback": text}


planner = real_planner_call if llm_client.HAS_KEY else fake_planner_brain
subagent_call = real_subagent_brain_call if llm_client.HAS_KEY else None  # resolved per-subtask below
critic = real_critic_call if llm_client.HAS_KEY else fake_mike
print(f"Using {'real model calls' if llm_client.HAS_KEY else 'mock brains'} for Jack, Bob's subagents, and Mike.")

In [5]:
def run_team(task, required_facts, planner_fn, critic_fn, tracer, verbose=True):
    tracer.record("Jack (planner)", duration_ms=random.uniform(300, 500), role="planner")
    subtasks = planner_fn(task)
    if verbose:
        print(f"[Jack] Plan: {subtasks}")

    notes = []
    for subtask in subtasks:
        tracer.record("Bob -> subagent", duration_ms=random.uniform(250, 450), subtask=subtask)
        if llm_client.HAS_KEY:
            synthesis = real_subagent_brain_call(subtask)
        else:
            synthesis, _ = run_subagent(subtask, subagent_brain, SUBAGENT_TOOLS)
        if verbose:
            print(f"[Bob's subagent] {subtask!r} -> {synthesis!r}")
        notes.append(synthesis)
    draft = " ".join(notes)
    if verbose:
        print(f"[Bob] Draft: {draft}")

    tracer.record("Mike (critic)", duration_ms=random.uniform(200, 350), role="critic")
    review = critic_fn(draft, required_facts)
    if verbose:
        print(f"[Mike] Verdict: {review}")

    return draft, review


TASK = "Prepare a short knowledge brief: who founded Anthropic, and what is the ReAct pattern?"
REQUIRED_FACTS = ["anthropic", "react"]

tracer = Tracer()
draft, review = run_team(TASK, REQUIRED_FACTS, planner, critic, tracer)

print("\n=== TRACE ===")
tracer.print_trace()
print(f"\nTotal latency: {tracer.total_ms():.1f}ms")


[Jack] Plan: ['who founded anthropic?', 'what is the react pattern?']
[Bob's subagent] 'who founded anthropic?' -> 'Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.'
[Bob's subagent] 'what is the react pattern?' -> 'ReAct interleaves reasoning traces with actions, letting a model plan and use tools within the same loop (Yao et al., 2022).'
[Bob] Draft: Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI. ReAct interleaves reasoning traces with actions, letting a model plan and use tools within the same loop (Yao et al., 2022).
[Mike] Verdict: {'verdict': 'approved', 'feedback': 'Covers everything required.'}

=== TRACE ===
[Jack (planner)] 427.9ms {'role': 'planner'}
[Bob -> subagent] 255.0ms {'subtask': 'who founded anthropic?'}
[Bob -> subagent] 305.0ms {'subtask': 'what is the react pattern?'}
[Mike (critic)] 233.5ms {'ro

## Break it #1: the same loop failure, now with a reusable guard

Chapter 1 fixed a straight-line infinite loop with duplicate-observation detection, built
directly into that chapter's loop. This time, Bob dispatches a subagent that hits the same
bait-tool bug, but the fix here is different on purpose: a max-iteration guard built as
reusable code, wrapping *any* brain function, rather than hardcoded into one loop. The two
techniques are complementary, not competing; a real system uses both.


In [6]:
def bait_tool(_input):
    return {"status": "partial", "detail": "Still processing your request, check back."}


def looping_subagent_brain(messages):
    observations = [m for m in messages if m["role"] == "observation"]
    if not observations or observations[-1]["content"].get("status") == "partial":
        return {"action": "bait_tool", "action_input": "any"}
    return {"action": "final_answer", "action_input": "done"}


print("--- Bug: Bob's subagent has no stop condition ---\n")
buggy_result, buggy_iterations = run_subagent(
    "generate the quarterly report subtask",
    looping_subagent_brain,
    tools={"bait_tool": bait_tool},
    max_iterations=15,  # capped ONLY so this cell terminates -- a real system has no such cap
    verbose=True,
)
print(f"\nUsed all {buggy_iterations} iterations without finishing.")


--- Bug: Bob's subagent has no stop condition ---

  [subagent step 1] {'action': 'bait_tool', 'action_input': 'any'}
  [subagent step 1] observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
  [subagent step 2] {'action': 'bait_tool', 'action_input': 'any'}
  [subagent step 2] observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
  [subagent step 3] {'action': 'bait_tool', 'action_input': 'any'}
  [subagent step 3] observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
  [subagent step 4] {'action': 'bait_tool', 'action_input': 'any'}
  [subagent step 4] observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
  [subagent step 5] {'action': 'bait_tool', 'action_input': 'any'}
  [subagent step 5] observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
  [subagent step 6] {'action': 'bait_tool', 'action_input':

In [7]:
def with_max_iterations(brain_fn, max_iterations: int = 3):
    '''Reusable guard: wraps ANY brain function so a run relying on it can never take more
    than max_iterations actions, regardless of which loop it's plugged into -- a simple,
    blunt backstop, distinct from (and complementary to) Chapter 1's duplicate-observation
    detection.'''
    def guarded(messages):
        iterations_so_far = sum(1 for m in messages if m["role"] == "observation")
        if iterations_so_far >= max_iterations:
            return {
                "action": "final_answer",
                "action_input": f"[guard] stopped after {max_iterations} iterations with no progress",
            }
        return brain_fn(messages)
    return guarded


print("--- Fix: same buggy brain, wrapped with a reusable max-iteration guard ---\n")
guarded_brain = with_max_iterations(looping_subagent_brain, max_iterations=3)
fixed_result, fixed_iterations = run_subagent(
    "generate the quarterly report subtask",
    guarded_brain,
    tools={"bait_tool": bait_tool},
    max_iterations=15,
    verbose=True,
)
print(f"\nStopped after {fixed_iterations} iterations (vs {buggy_iterations} before the guard).")
print("Result:", fixed_result)


--- Fix: same buggy brain, wrapped with a reusable max-iteration guard ---

  [subagent step 1] {'action': 'bait_tool', 'action_input': 'any'}
  [subagent step 1] observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
  [subagent step 2] {'action': 'bait_tool', 'action_input': 'any'}
  [subagent step 2] observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
  [subagent step 3] {'action': 'bait_tool', 'action_input': 'any'}
  [subagent step 3] observation: {'status': 'partial', 'detail': 'Still processing your request, check back.'}
  [subagent step 4] {'action': 'final_answer', 'action_input': '[guard] stopped after 3 iterations with no progress'}

Stopped after 4 iterations (vs 15 before the guard).
Result: [guard] stopped after 3 iterations with no progress


## Break it #2: Jack and Mike stuck in a cycle

A genuine cycle, distinct from a straight-line loop: Mike keeps asking Jack to clarify scope;
Jack keeps asking Mike what's unclear. Neither message is byte-identical to the one before
it, since each rephrases slightly, so a naive duplicate-hash check (comparing only to the
immediately preceding message) will not catch this. We'll demonstrate that failure
explicitly, then fix it with a similarity-based cycle detector.


In [8]:
JACK_VARIANTS = [
    "Can you clarify what's unclear about the scope, Mike?",
    "Could you clarify what's unclear about the scope, Mike?",
    "Can you please clarify what's unclear about the scope, Mike?",
]
MIKE_VARIANTS = [
    "Ask Jack to clarify the scope before I can review it.",
    "Please ask Jack to clarify the scope before I review it.",
    "Ask Jack to clarify the scope before this can be reviewed.",
]


def cycling_jack(history):
    round_idx = sum(1 for who, _ in history if who == "Jack")
    return JACK_VARIANTS[round_idx % len(JACK_VARIANTS)]


def cycling_mike(history):
    round_idx = sum(1 for who, _ in history if who == "Mike")
    return MIKE_VARIANTS[round_idx % len(MIKE_VARIANTS)]


def detect_cycle(history, lookback=2, threshold=0.6):
    '''Compares each speaker's latest messages to each other (not the other speaker's) using
    difflib's string similarity ratio -- a "simple string similarity" cycle detector, no API
    needed. A straight-line hash check compares only to the immediately preceding message,
    which is a DIFFERENT speaker's text and so never matches; this compares same-speaker
    turns to each other and tolerates paraphrasing, which is what actually catches a stuck
    back-and-forth.'''
    by_speaker = {}
    for speaker, msg in history:
        by_speaker.setdefault(speaker, []).append(msg)
    for msgs in by_speaker.values():
        if len(msgs) < lookback:
            continue
        recent = msgs[-lookback:]
        if all(
            difflib.SequenceMatcher(None, recent[i], recent[i + 1]).ratio() >= threshold
            for i in range(len(recent) - 1)
        ):
            return True
    return False


def run_conversation(jack_fn, mike_fn, max_rounds=8, duplicate_guard=False, cycle_guard=False, verbose=True):
    history = []
    last_hash = None
    for round_num in range(1, max_rounds + 1):
        jack_msg = jack_fn(history)
        history.append(("Jack", jack_msg))
        if verbose:
            print(f"[round {round_num}] Jack: {jack_msg}")
        if duplicate_guard:
            h = hash(jack_msg)
            if h == last_hash:
                return f"Stopped by duplicate-hash guard after round {round_num}.", history
            last_hash = h
        if cycle_guard and detect_cycle(history):
            return f"Stopped by semantic cycle guard after round {round_num}.", history

        mike_msg = mike_fn(history)
        history.append(("Mike", mike_msg))
        if verbose:
            print(f"[round {round_num}] Mike: {mike_msg}")
        if duplicate_guard:
            h = hash(mike_msg)
            if h == last_hash:
                return f"Stopped by duplicate-hash guard after round {round_num}.", history
            last_hash = h
        if cycle_guard and detect_cycle(history):
            return f"Stopped by semantic cycle guard after round {round_num}.", history

    return f"Reached max_rounds={max_rounds} without resolution.", history


print("--- Bug: hash-based duplicate check misses the cycle ---\n")
outcome, history = run_conversation(cycling_jack, cycling_mike, duplicate_guard=True)
print("\nOutcome:", outcome)


--- Bug: hash-based duplicate check misses the cycle ---

[round 1] Jack: Can you clarify what's unclear about the scope, Mike?
[round 1] Mike: Ask Jack to clarify the scope before I can review it.
[round 2] Jack: Could you clarify what's unclear about the scope, Mike?
[round 2] Mike: Please ask Jack to clarify the scope before I review it.
[round 3] Jack: Can you please clarify what's unclear about the scope, Mike?
[round 3] Mike: Ask Jack to clarify the scope before this can be reviewed.
[round 4] Jack: Can you clarify what's unclear about the scope, Mike?
[round 4] Mike: Ask Jack to clarify the scope before I can review it.
[round 5] Jack: Could you clarify what's unclear about the scope, Mike?
[round 5] Mike: Please ask Jack to clarify the scope before I review it.
[round 6] Jack: Can you please clarify what's unclear about the scope, Mike?
[round 6] Mike: Ask Jack to clarify the scope before this can be reviewed.
[round 7] Jack: Can you clarify what's unclear about the scope, Mike

In [9]:
print("--- Fix: semantic-similarity cycle detector catches it ---\n")
outcome2, history2 = run_conversation(cycling_jack, cycling_mike, cycle_guard=True)
print("\nOutcome:", outcome2)
print(f"\n(Bug ran the full max_rounds; the fix caught it after {len(history2)} messages.)")


--- Fix: semantic-similarity cycle detector catches it ---

[round 1] Jack: Can you clarify what's unclear about the scope, Mike?
[round 1] Mike: Ask Jack to clarify the scope before I can review it.
[round 2] Jack: Could you clarify what's unclear about the scope, Mike?

Outcome: Stopped by semantic cycle guard after round 2.

(Bug ran the full max_rounds; the fix caught it after 3 messages.)


## Break it #3: an unnecessary middle-manager role

Someone adds a second reviewer, Nina, after Mike, "just to be safe." It adds latency with no
functional value the moment Nina's verdict always matches Mike's. We'll build a latency
profiler that identifies exactly which hop added time with no corresponding value-add, plus a
cost/latency calculator comparing single-agent vs. multi-agent across task complexity.


In [10]:
def fake_second_reviewer(draft, required_facts):
    # Literally the same logic as fake_mike -- a redundant, duplicate check adding a role
    # that contributes no value on top of what Mike already does.
    return fake_mike(draft, required_facts)


tracer2 = Tracer()
draft2, mike_review = run_team(TASK, REQUIRED_FACTS, planner, critic, tracer2, verbose=False)
tracer2.record("Nina (second reviewer)", duration_ms=random.uniform(200, 350), role="redundant_reviewer")
nina_review = fake_second_reviewer(draft2, REQUIRED_FACTS)

print("Mike's verdict:", mike_review["verdict"])
print("Nina's verdict:", nina_review["verdict"])
print("Same verdict, zero new information -- Nina added latency with no value-add.\n")


def profile_hops(tracer, value_add_hop_names):
    return [
        {"name": s.name, "duration_ms": round(s.duration_ms, 1), "value_add": s.name in value_add_hop_names}
        for s in tracer.spans
    ]


profile = profile_hops(tracer2, value_add_hop_names={"Jack (planner)", "Bob -> subagent", "Mike (critic)"})
for row in profile:
    flag = "" if row["value_add"] else "  <-- no value-add"
    print(f"{row['name']:30s} {row['duration_ms']:>7.1f}ms{flag}")

no_value_total = sum(r["duration_ms"] for r in profile if not r["value_add"])
print(f"\nLatency with no corresponding value-add: {no_value_total:.1f}ms of {tracer2.total_ms():.1f}ms total.")


Mike's verdict: approved
Nina's verdict: approved
Same verdict, zero new information -- Nina added latency with no value-add.

Jack (planner)                   447.3ms
Bob -> subagent                  385.3ms
Bob -> subagent                  428.4ms
Mike (critic)                    213.0ms
Nina (second reviewer)           263.3ms  <-- no value-add

Latency with no corresponding value-add: 263.3ms of 1737.4ms total.


In [11]:
def estimate_cost_latency(num_agents, task_complexity, per_hop_latency_ms=350, per_hop_tokens=800, cost_per_million=5.0):
    '''Toy cost/latency model: N sequential hops, each costing per_hop_latency_ms and
    per_hop_tokens, scaled by a task-complexity multiplier.'''
    multiplier = {"simple": 1, "moderate": 2, "complex": 4}[task_complexity]
    total_latency_ms = num_agents * per_hop_latency_ms * multiplier
    total_tokens = num_agents * per_hop_tokens * multiplier
    total_cost = total_tokens / 1_000_000 * cost_per_million
    return {"latency_ms": total_latency_ms, "tokens": total_tokens, "cost_usd": round(total_cost, 4)}


print(f"{'complexity':10s} {'single-agent':40s} {'3-agent team':40s}")
for complexity in ["simple", "moderate", "complex"]:
    single = estimate_cost_latency(1, complexity)
    multi = estimate_cost_latency(3, complexity)
    print(f"{complexity:10s} {str(single):40s} {str(multi):40s}")


complexity single-agent                             3-agent team                            
simple     {'latency_ms': 350, 'tokens': 800, 'cost_usd': 0.004} {'latency_ms': 1050, 'tokens': 2400, 'cost_usd': 0.012}
moderate   {'latency_ms': 700, 'tokens': 1600, 'cost_usd': 0.008} {'latency_ms': 2100, 'tokens': 4800, 'cost_usd': 0.024}
complex    {'latency_ms': 1400, 'tokens': 3200, 'cost_usd': 0.016} {'latency_ms': 4200, 'tokens': 9600, 'cost_usd': 0.048}


## Break it #4: the leaky subagent

This is the context-isolation mechanic from the concept section, broken and then fixed
hands-on. Instead of returning just the final answer, imagine Bob's subagent returns its
*entire* raw transcript: every thought, every tool call, every observation. We built this
exact failure mode into `run_subagent()` already: `leaky=True`.


In [12]:
def count_words(text: str) -> int:
    '''A word count, not a real token count -- Chapter 5 introduces real tokenization with
    tiktoken and shows its quirks in depth. This is a deliberately dependency-free proxy for
    "how much context does this cost," good enough to demonstrate a size spike without
    requiring a network fetch just to measure it.'''
    return len(text.split())


compressed_result, _ = run_subagent("who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, leaky=False)
leaky_result, _ = run_subagent("who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, leaky=True)

print("Compressed synthesis (the correct behavior):")
print(" ", compressed_result)
print(f"  words: {count_words(compressed_result)}\n")

print("Leaky full transcript (the bug):")
print(" ", leaky_result)
print(f"  words: {count_words(leaky_result)}\n")

spike = count_words(leaky_result) / count_words(compressed_result)
print(f"Size spike: {spike:.1f}x more content entering Bob's context for the exact same underlying work.")
print("The correct answer is still in there -- but now it's one fact buried inside JSON")
print("scaffolding (roles, tool names, status fields) that Bob's own context has to carry")
print("forward on every subsequent call. Feed enough of this into a real model and the signal")
print("gets diluted by the noise, and every later call in the conversation costs more tokens")
print("for it, whether or not that specific noise is ever relevant again.")


Compressed synthesis (the correct behavior):
  Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI.
  words: 21

Leaky full transcript (the bug):
  [{"role": "user", "content": "who founded anthropic?"}, {"role": "observation", "tool": "mock_search", "content": {"status": "ok", "result": "Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI."}}, {"role": "assistant", "content": "Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with several colleagues who had previously worked at OpenAI."}]
  words: 59

Size spike: 2.8x more content entering Bob's context for the exact same underlying work.
The correct answer is still in there -- but now it's one fact buried inside JSON
scaffolding (roles, tool names, status fields) that Bob's own context has to carry
forward on every subsequent call. Feed enough of this int

### Detecting a leak, rather than just capping the size

`enforce_compression` below is a blunt instrument: it truncates anything over a word budget,
which bounds the damage but tells you nothing about *why* a result was oversized. Before
truncating, it's worth knowing whether you are looking at a leak at all.

Write that detector. The temptation is to reach for a length threshold, since the leaked
transcript in the cell above was visibly bigger. Resist it: size and leakage are independent.
A subagent answering a genuinely broad question returns a long, perfectly well-behaved
paragraph, and a subagent that leaks after one step returns a transcript shorter than that
paragraph. What actually distinguishes a leak is its *shape*: a serialized list of message
objects, each with a role and content, rather than prose.

In [ ]:
def is_leaky_result(raw_result: str) -> bool:
    '''Did a subagent hand back its raw transcript instead of a compressed synthesis?

    Decide on structure, not size. A leaked transcript is what run_subagent(leaky=True)
    produces: JSON that parses to a non-empty list whose every element is a dict with both
    a "role" and a "content" key.

    Anything else -- prose of any length, JSON scalars, an array of plain strings, an empty
    array, text that merely contains the word "role" -- is not a leak. Return a real bool.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


is_leaky_result = check("ch02-leak-check", is_leaky_result)

In [ ]:
def enforce_compression(raw_result: str, max_words: int = 30) -> str:
    '''The fix: a hard boundary Bob applies to ANYTHING a subagent hands back, so a leaky
    subagent can't silently pollute Bob's context even if it tries to.'''
    words = raw_result.split()
    if len(words) <= max_words:
        return raw_result
    return " ".join(words[:max_words]) + " [...TRUNCATED: subagent returned an oversized, uncompressed result]"


print("Leak detector, on the two results from the cell above:")
print(f"  compressed synthesis -> is_leaky_result = {is_leaky_result(compressed_result)}")
print(f"  leaky transcript     -> is_leaky_result = {is_leaky_result(leaky_result)}")
print()

print("--- Fix: Bob enforces a compression boundary on every subagent result ---\n")
safe_result = enforce_compression(leaky_result)
print("Bob actually receives:", safe_result)
print(f"  words: {count_words(safe_result)} (was {count_words(leaky_result)} before the boundary)")

## Bridge: what this looks like as a real framework, and as a real product

### As LangGraph

Jack/Bob/Mike maps cleanly onto LangGraph's nodes-and-edges model (LangChain, Inc.; see
`REFERENCES.md`). This sketch doesn't require the `langgraph` package (not a dependency of
this course); it's illustrative, not runnable LangGraph, but it accurately maps the concepts:
nodes are states, edges are transitions, and Mike's review is a conditional edge, the same
mechanism a state machine uses to loop back only when needed.


In [14]:
langgraph_sketch = {
    "nodes": {
        "jack_plan": "planner node -- produces subtasks",
        "bob_work": "worker node -- dispatches a subagent per subtask, compresses each result",
        "mike_review": "critic node -- approves or requests revision",
    },
    "edges": [
        ("jack_plan", "bob_work"),
        ("bob_work", "mike_review"),
        ("mike_review", "bob_work"),   # conditional edge -- only taken on "revise"
        ("mike_review", "END"),          # conditional edge -- only taken on "approved"
    ],
}

for node, description in langgraph_sketch["nodes"].items():
    print(f"node {node!r}: {description}")
print()
for src, dst in langgraph_sketch["edges"]:
    print(f"  {src:15s} -> {dst}")


node 'jack_plan': planner node -- produces subtasks
node 'bob_work': worker node -- dispatches a subagent per subtask, compresses each result
node 'mike_review': critic node -- approves or requests revision

  jack_plan       -> bob_work
  bob_work        -> mike_review
  mike_review     -> bob_work
  mike_review     -> END


### As OpenClaw

OpenClaw is the viral, self-hosted AI agent that runs continuously and connects through
WhatsApp, Telegram, Slack, and dozens of other channels. It uses a documented three-layer
architecture (channel, brain, body) plus a seven-stage loop (normalize, route, assemble
context, infer, ReAct, load skills, persist memory; see `docs.openclaw.ai` and
`REFERENCES.md`). It maps closely onto the planner/worker/critic split above:

| OpenClaw stage | Jack/Bob/Mike equivalent |
|---|---|
| Normalize (channel layer) | Not built in this notebook: whatever brings a task in (a Slack message, a support ticket) before Jack ever sees it. |
| Route | Deciding this task needs the full planner/worker/critic team at all, vs. a single-agent path. |
| Assemble context | Jack gathering what's needed before producing a plan. |
| Infer | The model call inside each of Jack, Bob's subagents, and Mike's individual decisions. |
| ReAct | Bob's subagent dispatch: each subagent runs its own Thought -> Action -> Observation loop, in isolation. |
| Load skills | Not built in this notebook, but this is exactly the skills concept from earlier in this chapter. OpenClaw's own skills layer is a real, shipped instance of it, not just a course abstraction. |
| Persist memory | Not built in this notebook (Chapter 4 covers staleness and cache invalidation in stored context). |

The honest caveat: this notebook's pipeline is a simplified teaching version. OpenClaw is a
real, continuously-running production system with all seven stages actually implemented. The
table above is a map, not a claim that this notebook reimplements OpenClaw.


## Interview preparation

### Recap

- Subagents isolate execution; skills isolate capability. A subagent's defining trait is a
  fresh, bounded context and a *compressed* return value, not just "another agent call."
- Supervisor-worker (Jack/Bob/Mike) is the current production default among the four
  subagent management patterns. Reach for fan-out, persistent pools, or peer-to-peer only
  when you have a specific reason to.
- A straight-line loop and a genuine cycle are different failure modes needing different
  detectors: duplicate-hash/max-iteration guards catch the former, while a similarity-based
  detector is needed for the latter, since no single message repeats verbatim.
- More agents means more latency and cost by default. That's only worth it when each added
  hop provides a value-add a single agent (or a shorter pipeline) couldn't.
- A leaky subagent measurably bloats the orchestrator's context (this chapter showed a
  concrete token-count spike). Enforce compression as a hard boundary; don't just hope
  subagents behave.

### Cold diagnosis exercise

For each symptom below, classify it as a loop, a cycle, an unnecessary hop, or a leaky
subagent before checking `solutions/ch02_control_flow_answers.md`. These are deliberately
phrased without naming Jack, Bob, or Mike, to make this an actual diagnosis exercise rather
than pattern-matching on names.

1. "An agent keeps calling the same tool over and over, and every response looks identical."
2. "Two agents keep sending each other slightly different messages, but neither ever makes
   progress; the conversation just keeps going."
3. "Adding a second reviewer step made latency worse, but the two reviewers always agree."
4. "After adding a subagent, the orchestrator's context ballooned in size and its next
   decision got noticeably worse."

### Architectural tradeoff questions

Attempt these from memory, then check `solutions/ch02_control_flow_answers.md`.

1. When is adding more agents actually a bad architectural decision?
2. What's the actual difference between a subagent and just calling another agent?
3. When would you reach for fan-out instead of a persistent agent pool?

Check your answers against `solutions/ch02_control_flow_answers.md`.


## Next: Chapter 3: RAG and Retrieval Evaluation

This chapter's team all shared one thing: they answered from a tiny, hand-built fact
dictionary. Chapter 3 replaces that with a real retrieval pipeline over real documents, and,
critically, shows exactly why grounding a model in retrieved context still doesn't eliminate
hallucination, hands-on.
